# REATS — Module B Baseline (Kaggle T4)
**ConvNeXt_tiny + Averaging Ensemble** | Target: Accuracy ≥ 92%, ECE ≤ 0.05

Paper: Do et al. (2025), JKSCI Vol.30 No.1

| Step | Cell | Description |
|---|---|---|
| 1 | GPU Check | Detect GPU, set auto batch_size |
| 2 | Install | kornia, grad-cam, shap, mlflow |
| 3 | Repo | Clone REATS branch, add to sys.path |
| 4 | Config | All hyper-parameters in one place |
| 5 | Dataset | Synthetic IR fallback OR real Kaggle dataset |
| 6 | Loaders | ImageFolder → DataLoader |
| 7 | Model | ConvNeXt_tiny + ConvNeXt build |
| 8 | Train | 300-epoch loop, checkpoint from epoch 225 |
| 9 | Evaluate | Test accuracy + ECE |
| 10 | Confusion | Per-class confusion matrix |
| 11 | Grad-CAM | XAI preview (6 samples) |
| 12 | Download | Checkpoint download instructions |

> ⚠ Before running: **Notebook settings → Accelerator → GPU T4 x1**


## 1 · GPU & Environment Check


In [ ]:
import torch, sys, platform

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'Platform: {platform.platform()}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    p = torch.cuda.get_device_properties(0)
    vram_gb = p.total_memory / 1e9
    print(f'\nGPU  : {p.name}')
    print(f'VRAM : {vram_gb:.1f} GB')
    BATCH_SIZE = 128 if vram_gb >= 20 else 64
    print(f'Auto batch_size → {BATCH_SIZE}  (128 for A100, 64 for T4)')
else:
    print('⚠ No GPU detected — enable in Notebook settings → Accelerator')
    BATCH_SIZE = 16

## 2 · Install Dependencies


In [ ]:
import subprocess, sys
pkgs = [
    'kornia>=0.7.0',
    'grad-cam>=1.4.8',
    'shap>=0.44.0',
    'mlflow>=2.10.0',
    'torchmetrics>=1.0.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('All packages installed.')

## 3 · Clone REATS Repository


In [ ]:
import subprocess, sys
from pathlib import Path

REPO   = 'https://github.com/trinhvutuandat34/Real-time-Explainable-Automatic-Target-Recognition-System.git'
BRANCH = 'claude/hopeful-mendel-x2y9j7'
TARGET = Path('/kaggle/working/reats')

if TARGET.exists():
    print('Repo exists — pulling latest...')
    subprocess.check_call(['git', '-C', str(TARGET), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(TARGET), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(TARGET), 'pull', 'origin', BRANCH])
else:
    subprocess.check_call(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, str(TARGET)])

# Force-evict any previously cached REATS modules so the pulled version loads fresh
for _k in [k for k in sys.modules if 'module_b' in k or 'module_c' in k or 'module_a' in k]:
    del sys.modules[_k]

if str(TARGET / 'REATS') not in sys.path:
    sys.path.insert(0, str(TARGET / 'REATS'))

from modules.module_b_classifier import (
    KorniaAugmentPipeline, build_convnext, EnsembleClassifier,
    build_loaders, train_one_epoch, evaluate, compute_ece, CLASSES,
)
from modules.module_c_xai import GradCAMExplainer
print('REATS modules loaded OK')

## 4 · Configuration


In [ ]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────
DATA_DIR = Path('/kaggle/working/reats/REATS/data')
CKPT_DIR = Path('/kaggle/working/checkpoints')
RUNS_DIR = Path('/kaggle/working/mlruns')
CKPT_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)

# ── Dataset source ─────────────────────────────────────────────────────
# Priority (auto-detected in Cell 5):
#   1. FLIR ADAS thermal dataset (deepnewbie/flir-thermal-images-dataset)
#   2. Real REATS Kaggle dataset (set KAGGLE_DATASET to your slug)
#   3. Synthetic-only fallback (no data files required)
FLIR_DIR       = Path('/kaggle/input/flir-thermal-images-dataset')  # mount as Kaggle dataset
KAGGLE_DATASET = None   # e.g. 'reats-ir-images' — your own labelled dataset slug

# ── Training hyper-parameters ───────────────────────────────────────────
EPOCHS           = 300   # paper: 300
BEST_EPOCH_START = 225   # paper: save only from epoch 225
LR               = 1e-4  # paper: AdamW lr=1e-4
NUM_CLASSES      = 6
IMG_SIZE         = 224

CONFIG = dict(
    data_root        = str(DATA_DIR),
    num_classes      = NUM_CLASSES,
    img_size         = IMG_SIZE,
    batch_size       = BATCH_SIZE,
    lr               = LR,
    epochs           = EPOCHS,
    best_epoch_start = BEST_EPOCH_START,
    device           = device,
    classes          = CLASSES,
)
print('Config:')
for k, v in CONFIG.items():
    print(f'  {k:<20} = {v}')

## 5 · Dataset

Three sources are tried in priority order — the first available wins:

| Priority | Source | How to enable |
|---|---|---|
| 1 | **FLIR ADAS thermal** | Add `deepnewbie/flir-thermal-images-dataset` as a Kaggle input dataset |
| 2 | **Own REATS dataset** | Set `KAGGLE_DATASET` to your labelled-images slug |
| 3 | **Synth-only** | Automatic fallback — no data files needed |

The generator script (`generate_flir_fallback.py`) handles all three paths:
- **FLIR mode**: crops annotated bboxes, intensity-remaps to per-class thermal signature, adds Gaussian target blob
- **Synth-only**: generates plausible IR-like backgrounds + shaped hot target
- Idempotent — re-runs only fill the shortfall toward **170 train / 30 val / 200 test** per class


In [ ]:
import subprocess
from pathlib import Path

TARGET = Path('/kaggle/working/reats')   # same as Cell 3 — safe to re-define
FALLBACK_SCRIPT = TARGET / 'REATS' / 'generate_flir_fallback.py'

def _run_fallback(mode, flir_arg=None):
    cmd = ['python', str(FALLBACK_SCRIPT), '--out', str(DATA_DIR), '--mode', mode]
    if flir_arg:
        cmd += ['--flir', str(flir_arg)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
    return r.returncode == 0

# ── Priority 1: FLIR ADAS dataset ───────────────────────────────────────
if FLIR_DIR.exists() and any(FLIR_DIR.iterdir()):
    print(f'✓ FLIR ADAS found: {FLIR_DIR}')
    print('  Running crop + background modes...')
    ok = _run_fallback('both', flir_arg=FLIR_DIR)
    if not ok:
        print('  FLIR mode failed — falling back to synth-only')
        _run_fallback('synth-only')

# ── Priority 2: labelled Kaggle dataset ─────────────────────────────────
elif KAGGLE_DATASET:
    src = Path(f'/kaggle/input/{KAGGLE_DATASET}')
    assert src.exists(), f'{src} not found — check KAGGLE_DATASET slug'
    import shutil
    print(f'Copying real dataset from {src} ...')
    for split in ['train', 'val', 'test']:
        for cls in CLASSES:
            s = src / split / cls
            if s.exists():
                d = DATA_DIR / split / cls
                d.mkdir(parents=True, exist_ok=True)
                for img in s.iterdir():
                    shutil.copy2(img, d / img.name)
    print('Done copying.')

# ── Priority 3: pure synthetic fallback ─────────────────────────────────
else:
    print('No FLIR dataset found — using synth-only fallback')
    print('  (Add deepnewbie/flir-thermal-images-dataset for higher-quality data)')
    _run_fallback('synth-only')

In [ ]:
import subprocess
r = subprocess.run(
    ['python', str(TARGET / 'REATS' / 'dataset_validator.py'), '--root', str(DATA_DIR)],
    capture_output=True, text=True,
)
print(r.stdout)

### Sample images from each class


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, len(CLASSES), figsize=(18, 3))
for ax, cls in zip(axes, CLASSES):
    imgs = sorted((DATA_DIR / 'train' / cls).glob('*'))[:1]
    if imgs:
        ax.imshow(Image.open(imgs[0]), cmap='gray')
    ax.set_title(cls, fontsize=10)
    ax.axis('off')
plt.suptitle('Sample IR images — train set', fontsize=12)
plt.tight_layout()
plt.show()

## 6 · Data Loaders


In [ ]:
train_loader, val_loader, test_loader = build_loaders(CONFIG)

print(f'Train: {len(train_loader.dataset):5d} images  {len(train_loader):3d} batches')
print(f'Val  : {len(val_loader.dataset):5d} images  {len(val_loader):3d} batches')
print(f'Test : {len(test_loader.dataset):5d} images  {len(test_loader):3d} batches')
print(f'Batch size: {BATCH_SIZE}')

## 7 · Model


In [ ]:
import torch.nn as nn

model        = build_convnext(num_classes=NUM_CLASSES).to(device)
optimizer    = torch.optim.AdamW(model.parameters(), lr=LR)
criterion    = nn.CrossEntropyLoss()
aug_pipeline = KorniaAugmentPipeline().to(device)

CKPT_PATH    = CKPT_DIR / 'convnext_best.pth'
start_epoch  = 1
best_val_acc = 0.0

if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=device)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        model.load_state_dict(ckpt['state_dict'])
        start_epoch  = ckpt.get('epoch', 1) + 1
        best_val_acc = ckpt.get('best_val_acc', 0.0)
    else:
        model.load_state_dict(ckpt)
    print(f'Resumed  epoch={start_epoch - 1}  best_val_acc={best_val_acc:.4f}')
else:
    print('No checkpoint — training from scratch')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'ConvNeXt_tiny trainable params: {n_params:.1f}M')

## 8 · Training

- Augmentation runs on GPU via Kornia (each transform p=0.5, per Table 1 of the paper)
- Checkpoint saved to `checkpoints/convnext_best.pth` whenever val_acc improves
- Also saves a periodic backup every 25 epochs to survive Kaggle session limits
- MLflow logs to `/kaggle/working/mlruns/`


In [ ]:
import mlflow, time, os

# MLflow ≥2.x dropped the filesystem store — use SQLite instead
MLFLOW_DB = RUNS_DIR / 'mlflow.db'
mlflow.set_tracking_uri(f'sqlite:///{MLFLOW_DB}')
mlflow.set_experiment('REATS-Baseline')

epochs_left = EPOCHS - start_epoch + 1
print(f'Training {epochs_left} epoch(s)  [{start_epoch}..{EPOCHS}]')
print(f'Validation + checkpointing from epoch {BEST_EPOCH_START}')
print(f'Checkpoint path: {CKPT_PATH}\n')

with mlflow.start_run(run_name='ConvNeXt_tiny_baseline'):
    mlflow.log_params(CONFIG)

    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, aug_pipeline, device,
        )
        mlflow.log_metrics({'train_loss': tr_loss, 'train_acc': tr_acc}, step=epoch)

        val_note = ''
        if epoch >= BEST_EPOCH_START:
            val_loss, val_acc = evaluate(model, val_loader, criterion, device)
            mlflow.log_metrics({'val_loss': val_loss, 'val_acc': val_acc}, step=epoch)
            val_note = f'  val={val_acc:.4f}'
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(
                    {'state_dict': model.state_dict(),
                     'epoch': epoch,
                     'best_val_acc': best_val_acc},
                    CKPT_PATH,
                )
                val_note += ' ← BEST'

        # Periodic backup every 25 epochs (insurance against session timeout)
        if epoch % 25 == 0:
            torch.save(
                {'state_dict': model.state_dict(), 'epoch': epoch},
                CKPT_DIR / f'convnext_ep{epoch:03d}.pth',
            )

        elapsed = time.time() - t0
        if epoch % 10 == 0 or val_note:
            print(f'[{epoch:3d}/{EPOCHS}]  loss={tr_loss:.4f}  acc={tr_acc:.4f}{val_note}  ({elapsed:.0f}s)')

target_ok = '✓ PASS' if best_val_acc >= 0.92 else ('~ CLOSE' if best_val_acc >= 0.90 else '✗ FAIL')
print(f'\nBest val acc: {best_val_acc:.4f}  {target_ok}  (target ≥ 0.92)')
mlflow.log_metric('best_val_acc', best_val_acc)

## 9 · Evaluation — Test Set


In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['state_dict'] if 'state_dict' in ckpt else ckpt)
model.eval()

_, test_acc = evaluate(model, test_loader, criterion, device)
ece         = compute_ece(model, test_loader, device)

def _check(val, target, op='ge'):
    ok = val >= target if op == 'ge' else val <= target
    return '✓ PASS' if ok else '✗ FAIL'

print(f'Test Accuracy : {test_acc:.4f}  {_check(test_acc, 0.92)}  (target ≥ 0.92)')
print(f'ECE           : {ece:.4f}   {_check(ece, 0.05, "le")}  (target ≤ 0.05)')

## 10 · Confusion Matrix


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(device)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — Test Set  (acc={test_acc:.3f})')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()
print('Fighter confusion (hardest per paper):')
fighter_idx = [CLASSES.index(c) for c in ['F16','MiG19','MiG21']]
for i in fighter_idx:
    row = cm[i, fighter_idx]
    print(f'  {CLASSES[i]:<6} →  ' + '  '.join(f'{CLASSES[j]}:{v}' for j, v in zip(fighter_idx, row)))

## 11 · Grad-CAM Preview (Module C)

Quick sanity check — the heatmap should light up the target region, not the background.


In [ ]:
import cv2, numpy as np

try:
    gradcam = GradCAMExplainer(model)
    sample_imgs, sample_labels = next(iter(test_loader))
    sample_imgs = sample_imgs[:6].to(device)

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    model.eval()
    for i in range(6):
        x       = sample_imgs[i:i+1]
        pred    = model(x).argmax(1).item()
        heatmap = gradcam.explain(x, pred)

        # Denormalise image: mean=0.5 std=0.5
        orig = (sample_imgs[i].cpu().permute(1,2,0).numpy() * 0.5 + 0.5)
        orig = (orig * 255).clip(0,255).astype(np.uint8)
        gray = orig[..., 0] if orig.ndim == 3 else orig

        hmap  = cv2.applyColorMap((heatmap*255).astype(np.uint8), cv2.COLORMAP_JET)
        hmap  = cv2.cvtColor(hmap, cv2.COLOR_BGR2RGB)
        over  = cv2.addWeighted(cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB), 0.6, hmap, 0.4, 0)

        axes[0,i].imshow(gray, cmap='gray')
        axes[0,i].set_title(f'True: {CLASSES[sample_labels[i]]}', fontsize=9)
        axes[0,i].axis('off')
        axes[1,i].imshow(over)
        axes[1,i].set_title(f'Pred: {CLASSES[pred]}', fontsize=9)
        axes[1,i].axis('off')

    fig.suptitle('Grad-CAM — top row: input  |  bottom row: attention heatmap', fontsize=11)
    plt.tight_layout()
    plt.savefig('/kaggle/working/gradcam_preview.png', dpi=150)
    plt.show()
    print('Saved: /kaggle/working/gradcam_preview.png')
except Exception as e:
    print(f'Grad-CAM error: {e}')

## 12 · Download Artifacts & Next Steps


In [ ]:
import os
print('━'*50)
print('  Artifacts saved in /kaggle/working/')
print('━'*50)
for p in sorted(Path('/kaggle/working').rglob('*')):
    if p.is_file() and p.suffix in {'.pth', '.png', '.yaml'}:
        size_kb = p.stat().st_size / 1024
        print(f'  {str(p.relative_to("/kaggle/working")):45s}  {size_kb:7.1f} KB')
print('━'*50)
print()
print('⚠  /kaggle/working/ is wiped when the session ends.')
print('   Download checkpoints/convnext_best.pth BEFORE closing.')
print()
print('Next steps:')
print('  • acc ≥ 0.92  → Done — commit checkpoint, move to Module A')
print('  • acc 0.90-0.92 → Train 6 models → EnsembleClassifier (paper: +2pp)')
print('  • acc < 0.90  → Check data quality, increase epochs')